# Solution C2 · Wrong labels: boosting chases them, the forest does not

**Type**: Break the model

## Objective

Corrupt 20% of the training labels and measure the accuracy on a **clean** test set as a function of the XGBoost rounds (learning rate 0.1, `max_depth=5`): it rises up to about 25 rounds and then drops by **about 6 points** while a random forest stays stable. Then show that regularization (`min_child_weight`,
`reg_lambda`, `subsample`, shallow trees) limits the damage.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(3000, n_features=10, n_informative=5, class_sep=1.2, flip_y=0, random_state=2)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
rng = np.random.default_rng(0)
y_noisy = np.where(rng.random(len(y_tr)) < 0.2, 1 - y_tr, y_tr)
ks = [10, 25, 50, 100, 250, 500]

## Commented solution

In [ ]:
def curve(**kw):
    m = xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, eval_metric="logloss", n_jobs=-1, random_state=0, **kw).fit(X_tr, y_noisy)
    return pd.Series({k: float((m.predict(X_te, iteration_range=(0, k)) == y_te).mean()) for k in ks})


xgb_acc = curve(max_depth=5)
reg_xgb_acc = curve(max_depth=3, min_child_weight=20, reg_lambda=10, subsample=0.7)
rf_acc = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X_tr, y_noisy).score(X_te, y_te)
clean_xgb_acc = float((xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr).predict(X_te) == y_te).mean())
print(pd.DataFrame({"xgb": xgb_acc, "regularized xgb": reg_xgb_acc}).round(3).T)
print(f"random forest: {rf_acc:.3f} | XGBoost with clean labels at 500 rounds: {clean_xgb_acc:.3f}")

In [ ]:
xgb_peak, peak_k = float(xgb_acc.max()), int(xgb_acc.idxmax())
xgb_loss = xgb_peak - xgb_acc[500]
reg_loss = float(reg_xgb_acc.max() - reg_xgb_acc[500])
xgb_beaten_by_rf = bool(rf_acc > xgb_acc[500])
rf_beats_xgb_at_peak = bool(rf_acc > xgb_peak)
print(f"XGBoost: peak {xgb_peak:.3f} at {peak_k} rounds, then loses {xgb_loss:.3f} | regularized: loses {reg_loss:.3f} | the forest ({rf_acc:.3f}) beats XGBoost at 500 rounds: {xgb_beaten_by_rf}; at the peak: {rf_beats_xgb_at_peak}")

## Check

In [ ]:
with verify():
    check(list(xgb_acc.index) == ks and list(reg_xgb_acc.index) == ks, "curves over six values of k", "iteration_range=(0, k)")
    check(clean_xgb_acc > 0.93, "with clean labels XGBoost reaches over 0.93", "y_tr instead of y_noisy")
    check(peak_k in (25, 50, 100) and xgb_peak > 0.89 and xgb_loss > 0.04 and xgb_acc[500] < xgb_acc[peak_k] - 0.04, "with noisy labels XGBoost has a peak at 25-100 rounds and then loses over 4 points", "xgb_acc.max() and xgb_acc[500]")
    check(reg_loss < xgb_loss - 0.02 and reg_xgb_acc.max() > 0.9, "the regularized model loses less than the unregularized one (by over 2 points) and goes over 0.9 at the peak", "min_child_weight, reg_lambda, subsample, depth 3")
    check(rf_acc > 0.9 and xgb_beaten_by_rf is True, "the forest (over 0.9) beats the unregularized XGBoost at 500 rounds", "rf_acc > xgb_acc[500]")
    check(rf_beats_xgb_at_peak in (True, False), "boolean answer (the forest and the XGBoost peak are close)", "compare rf_acc and xgb_peak")
done()

## What you learned

- **Boosting corrects the errors of the previous ones, even those that are not errors:** with 20% of the labels swapped the XGBoost accuracy rises up to 0.90 (25 rounds) and then drops to 0.85 at 500: the later rounds learn the wrong labels. With clean labels the same model reaches 0.95.
- **The forest is much more robust:** 0.915, above the boosting peak (0.904), **without** having to choose anything. Averaging independent trees is a good antidote to label noise; boosting needs a brake (early stopping, regularization).
- **Regularization helps a lot:** `max_depth=3`, `min_child_weight=20`, `reg_lambda=10`, `subsample=0.7` bring the peak to 0.92 and the loss at 500 rounds from 5.7 to 3.7 points. A high `min_child_weight` prevents isolating a few noisy examples; `subsample` decorrelates.
- **Warning sign in practice:** if your boosting model does **better with few rounds and shallow trees** than with many, suspect noise in the labels. A check: compare with a forest (M8): if the forest beats the "pushed" boosting, the signal is noisy.

## Common pitfall

Believing that "boosting is always the best" and pushing rounds and depth until the training is fine. With noisy data boosting is the model most at risk of overfitting; a forest or a heavily regularized boosting are safer choices.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How does boosting behave with noisy labels?</summary>

Worse than the forest, if not regularized: each round corrects the errors of the previous ones, so with enough rounds it tries to explain the wrong labels too and the error on clean data climbs back. It is mitigated with early stopping, shallow trees, high min_child_weight and reg_lambda, subsample and a low learning rate; and it is worth comparing it with a forest,
which is naturally robust to label noise.

</details>